# EDA RC Gas

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from statsmodels.tsa.stattools import acf

In [ ]:
BASE_DIR = Path.cwd()  # notebooks/
PROJECT_ROOT = BASE_DIR.parent  # notebooks/ -> raíz del repo
sys.dont_write_bytecode = True  # no crear __pycache__ en data_utils/
sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
from data_utils.fancy_barplot import fancy_bars
from data_utils.fancy_hist import fancy_hist
from data_utils.triple_plot import normality_report

In [ ]:
pd.set_option("display.width", 220)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="white")
INK, GRID = "#1F2E3D", "#E2E8F0"

# ---------------- Parámetros ----------------
NOMBRE_EXCEL = "Base de datos"
HOJA, POSICION_HOJA = "RC GAS", 1
RANDOM_STATE = 42
TEST_SIZE = 0.20

COL_TIEMPO, ACUM, DIF = "t_stamp", "V_Consumo_Gas_refinacion_Acumm", "Consumo_Diario"
ESQUEMA = {COL_TIEMPO: "datetime", ACUM: "float64", DIF: "float64"}
DIAS = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]

In [ ]:
HUECO_H = 30  # más de 30 h sin lectura = hueco
SABADO = 5  # dayofweek: 0 = lunes … 5 = sábado
MIN_N_GRUPO = 3  # tamaño mínimo de grupo para Kruskal-Wallis
Z_MAD_UMBRAL = 3.5

## 1. Carga de la hoja `RC GAS`

In [ ]:
RUTA = Path("../data/Base de datos.xlsx")
raw = pd.read_excel(RUTA, sheet_name=HOJA)
print(f"Libro: {RUTA.name}")
print(f"\nhojas: {pd.ExcelFile(RUTA).sheet_names}")
print(f"\nHoja cargada: '{HOJA}'")
print(f"\nforma: {raw.shape}")
dtypes_crudos = raw.dtypes.astype(str)

## 2. Split train/test
Antes de separar los datos, aplicamos transformaciones que requieren el dataset completo y ordenado cronológicamente para no romper la serie de tiempo. El proceso sigue estos pasos:

- Limpieza segura de datos: El código intenta convertir todas las lecturas a números. Si encuentra un texto o un formato inválido, no detiene el proceso, sino que deja la celda en blanco (errors="coerce") y guarda un registro de exactamente qué celdas fallaron para no perderles el rastro.

- Trazabilidad del archivo: Antes de organizar nada, se guarda el número de fila que cada dato tenía en el documento original (el CSV crudo). Esto funciona como una "etiqueta de rastreo" por si luego necesitamos auditar un dato raro directamente en la fuente.

- Cálculo de consumos y tiempos: Toda la información se ordena de la más antigua a la más reciente. Una vez ordenada la línea de tiempo, a cada fila se le calcula su consumo real (delta) restando la lectura anterior, y el tiempo transcurrido (paso_h) para saber cuántas horas pasaron entre una medición y otra.

**Split:** `sklearn.model_selection.train_test_split(test_size=0.2, random_state=42, shuffle=True)`.

> **Nota:** la hoja tiene solo 100 filas, así que el train (80) es pequeño y las conclusiones de estacionalidad son exploratorias. Para un modelo de pronóstico, usar `SHUFFLE=False` (split cronológico).

In [ ]:
X = raw.copy()
X["pos_original"] = np.arange(len(X))
fallo_cast = {}
X[COL_TIEMPO] = pd.to_datetime(X[COL_TIEMPO], errors="coerce")
fallo_cast[COL_TIEMPO] = X[COL_TIEMPO].isna() & raw[COL_TIEMPO].notna()
for c in (ACUM, DIF):
    num = pd.to_numeric(X[c], errors="coerce").astype("float64")
    fallo_cast[c] = num.isna() & raw[c].notna()
    X[c] = num
for c, s in fallo_cast.items():
    X[f"cast_falla__{c}"] = s

X = X.sort_values(COL_TIEMPO, kind="mergesort").reset_index(drop=True)
X["paso_h"] = X[COL_TIEMPO].diff().dt.total_seconds() / 3600
X["prev_acum"] = X[ACUM].shift(1)
X["delta"] = X[ACUM] - X["prev_acum"]

# Resolución según horas desde la lectura anterior:
##sub-horaria: por debajo de una hora
# intra-día: entre 1 y 20 horas, diaria (ideal): entre 20 y 30 horas
# hueco > 30 h: más de 30 horas
X["resolucion"] = pd.cut(
    X["paso_h"],
    [-np.inf, 0.05, 20, HUECO_H, np.inf],
    labels=["sub-horaria", "intra-día", "diaria", "hueco > 30 h"],
).astype(str)
X.loc[X["paso_h"].isna(), "resolucion"] = "primera lectura"

train, test = train_test_split(X, test_size=TEST_SIZE, random_state=RANDOM_STATE, shuffle=True)
train = train.sort_values(COL_TIEMPO).set_index(COL_TIEMPO)
n_test = len(test)
del test  # a partir de aquí el test no se vuelve a usar

print(f"train: {len(train)} filas · test (apartado): {n_test} filas")
print(f"train cubre {train.index.min()} → {train.index.max()}")

## 3. Validación de esquema (train)

In [ ]:
def chequeo(nombre: str, ok: bool, detalle: str) -> dict[str, str]:
    """Arma una fila de la tabla de validación de esquema."""
    return {"chequeo": nombre, "estado": "PASA" if ok else "FALLA", "detalle": detalle}


esperadas, presentes = set(ESQUEMA), set(raw.columns)
coherente = np.isclose(train[DIF], train["delta"], atol=0.11) | train["delta"].isna()
checks = [
    chequeo(
        "Columnas esperadas presentes",
        esperadas <= presentes,
        f"faltan: {sorted(esperadas - presentes) or '—'}",
    ),
    chequeo(
        "Sin columnas inesperadas",
        presentes <= esperadas,
        f"sobran: {sorted(presentes - esperadas) or '—'}",
    ),
    chequeo(
        "t_stamp parseable y sin nulos",
        train.index.notna().all() and not train[f"cast_falla__{COL_TIEMPO}"].any(),
        "",
    ),
    chequeo(
        "t_stamp único",
        not train.index.duplicated().any(),
        f"{int(train.index.duplicated().sum())} duplicados",
    ),
    chequeo(
        "Archivo en orden cronológico",
        train["pos_original"].is_monotonic_increasing,
        "orden original "
        + (
            "ascendente"
            if train["pos_original"].is_monotonic_increasing
            else "descendente"
            if train["pos_original"].is_monotonic_decreasing
            else "mezclado"
        ),
    ),
]
for c in (ACUM, DIF):
    checks.append(
        chequeo(
            f"{c}: numérico",
            not train[f"cast_falla__{c}"].any(),
            f"{int(train[f'cast_falla__{c}'].sum())} no convertibles",
        )
    )
checks += [
    chequeo(
        f"{ACUM}: lectura ≥ 0",
        (train[ACUM] >= 0).all(),
        f"{int((train[ACUM] < 0).sum())} negativos",
    ),
    chequeo(
        f"{ACUM}: acumulado no decreciente",
        (train["delta"].dropna() >= 0).all(),
        f"{int((train['delta'] < 0).sum())} filas con delta < 0",
    ),
    chequeo(
        f"{DIF} = diferencia del acumulado",
        coherente.all(),
        f"{int((~coherente).sum())} filas incoherentes",
    ),
    chequeo(f"{DIF} ≥ 0", (train[DIF] >= 0).all(), f"{int((train[DIF] < 0).sum())} negativos"),
]
esquema = pd.DataFrame(checks)
display(
    esquema.style.map(
        lambda v: "color:#b91c1c;font-weight:bold" if v == "FALLA" else "color:#15803d",
        subset=["estado"],
    )
)
print(f"Resultado: {(esquema.estado == 'PASA').sum()} de {len(esquema)} chequeos pasan")

## 4. Consistencia de tipos y conversiones necesarias (train)

In [ ]:
def _cumple(dtype_txt: str, esperado: str) -> bool:
    """Indica si el dtype leído coincide con el tipo lógico esperado."""
    if esperado == "datetime":
        return dtype_txt.startswith("datetime64")
    return dtype_txt == esperado


tipos = pd.DataFrame({"dtype_en_archivo": dtypes_crudos, "dtype_esperado": pd.Series(ESQUEMA)})
tipos["requiere_cast"] = [
    not _cumple(a, e)
    for a, e in zip(tipos["dtype_en_archivo"], tipos["dtype_esperado"], strict=True)
]
display(tipos)

dec = {
    c: train[c].dropna().map(lambda v: len(f"{v:.6f}".rstrip("0").split(".")[1])).max()
    for c in (ACUM, DIF)
}
print("Máximo de decimales observados:", dec, "→ el medidor reporta con resolución de 0,1 m^3")
print(
    "Precisión del timestamp: milisegundos; segundos de lectura alrededor de medianoche:",
    sorted(train.index[train.index.hour.isin([0, 23])].strftime("%H:%M:%S").unique())[:8],
    "…",
)

**Lectura:**
- Los tipos que infiere pandas ya son los esperados. Sin embargo,  en el pipeline de producción debemos forzar la conversión numérica explícitamente (errors="coerce"). Esto garantiza que si en el futuro llega un texto por error, el sistema lo convierta en un vacío seguro (NaN) en lugar de colapsar. 

- El medidor físico solo tiene sensibilidad para detectar cambios de 0,1 m^3. Por lo tanto, si vemos variaciones minúsculas de ±0,1 de una hora a otra, no debemos interpretarlas como un consumo real, sino como simple "ruido" o vibración natural del aparato.

- Como descubrimos que el reloj del SCADA no es exacto y registra el cierre del día unos segundos antes o después de las 12:00 (ej. 23:59:58 o 00:00:02), no podemos simplemente cortar la fecha usando la función normal (t_stamp.date()). Si lo hiciéramos, el sistema le asignaría por error la lectura al día de ayer. El código debe incluir una tolerancia de unos segundos para asignar lógicamente a qué día pertenece ese consumo.

## 5. Valores faltantes: conteos y patrones (train)

| Categoría | Regla |
|---|---|
| `nan_archivo` | celda vacía o no convertible |
| `primera_lectura` | no hay lectura previa → delta indefinido |
| `reinicio` | delta < 0 (el contador volvió a empezar) |
| `sin_cambio` | delta = 0 en una fila con resolución diaria (medidor detenido o sin flujo) |
| `tras_hueco` | la lectura previa está a más de 30 h → el delta mezcla varios días |
| `granularidad` | lectura sub-horaria o intra-día (no es un cierre diario) |
| `ok` | delta de un cierre diario consecutivo, ≥ 0 |

In [ ]:
t = train
flag = pd.Series("ok", index=t.index)
flag[t["resolucion"].isin(["sub-horaria", "intra-día"])] = (
    "granularidad"  # si el medidor reportó datos muy seguido
)
flag[t["resolucion"].eq("hueco > 30 h")] = "tras_hueco"
flag[t["delta"].eq(0) & t["resolucion"].eq("diaria")] = "sin_cambio"
flag[t["delta"].lt(0)] = "reinicio"
flag[t["delta"].isna()] = "primera_lectura"
flag[t[ACUM].isna() | t[f"cast_falla__{ACUM}"]] = "nan_archivo"
t = t.assign(categoria=flag)

print("NaN explícitos por columna (archivo):")
display(raw.loc[:, list(ESQUEMA)].isna().sum().to_frame("nulos_en_archivo").T)
conteo = flag.value_counts().to_frame("filas")
conteo["%"] = conteo["filas"] / len(flag) * 100
display(conteo)
fancy_bars(
    t.reset_index(),
    "categoria",
    labels="both",
    title="Categoría de cada fila (train)",
    figsize=(9, 4.5),
    rot=20,
)
plt.show()

- Podemos obserccar que el 40% de las filas está clasificado en granularidad, lo que nos muesra que casi la mitad de registros son variaciones minúsculas.
- Vemos que este medidor de gas no sufre de apagones masivos ni de congelamientos del SCADA.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 4.5))
colores = {
    "ok": "#5BA7DE",
    "granularidad": "#96A5A5",
    "tras_hueco": "#F0B26B",
    "reinicio": "#E28585",
    "sin_cambio": "#9B8FC7",
    "primera_lectura": "#1F2E3D",
    "nan_archivo": "#000000",
}
ax.plot(t.index, t[ACUM], color="#cbd5e1", lw=1, zorder=1)
for cat, g in t.groupby("categoria"):
    ax.scatter(g.index, g[ACUM], s=28, color=colores[cat], label=cat, zorder=2)
ax.set_title(
    "Lectura acumulada (train) coloreada por categoría", loc="left", fontweight="bold", color=INK
)
ax.legend(frameon=False, ncol=7)
ax.grid(axis="y", color=GRID, ls="--")
sns.despine(left=True)
plt.show()
display(
    t.loc[
        t.categoria.isin(["reinicio", "tras_hueco", "sin_cambio"]),
        [ACUM, DIF, "delta", "paso_h", "categoria"],
    ]
)

**Patrones:**
- No hay NaN explícitos, el faltante es implícito; Aparece como reinicios del contador, días sin cierre (huecos) y frecuencias de lectura irregulares.

- Las caídas a ~0 del 11 y el 30 de agosto no siguen un patrón de calendario y ocurren en valores distintos (17.143,2 y 14.427,2 m^3), así que  apuntan a reinicios manuales o programados. El consumo de esos días queda subestimado. Además, el 12 de agosto el medidor no cambia (`sin_cambio`): puede ser una parada de la planta o una falla temporal del sensor, lo cual hay que validar.

- Las lecturas por minuto y las ráfagas alrededor de la medianoche se concentran en los tres primeros días de la serie (1–3 de agosto). Al depender del periodo, el faltante no es aleatorio (MAR). Esto sugiere que corresponde a la fase de instalación y pruebas en el SCADA (hipótesis a confirmar con la planta).

## 6. Integridad temporal y frecuencia (train)

In [ ]:
integridad = pd.Series(
    {
        "inicio": t.index.min(),
        "fin": t.index.max(),
        "días calendario cubiertos": t.index.normalize().nunique(),
        "timestamps duplicados": int(t.index.duplicated().sum()),
        "paso mediano (h)": round(t["paso_h"].median(), 2),
        "filas con hueco > 30 h": int((t["paso_h"] > HUECO_H).sum()),
    }
)
display(integridad.to_frame("valor"))
print("resumen de resoluciones de paso_h")
display(t.groupby("resolucion")["paso_h"].agg(["count", "min", "median", "max"]))

lect_dia = t.groupby(t.index.normalize()).size()
fig, ax = plt.subplots(figsize=(15, 3.5))
ax.bar(lect_dia.index, lect_dia.values, color="#5BA7DE")
ax.set_title("Lecturas por día calendario (train)", loc="left", fontweight="bold", color=INK)
sns.despine(left=True)
plt.show()
huecos = t[t["paso_h"] > HUECO_H]
print("Huecos (> 30 h) en la serie completa detectados desde train:")
display(
    pd.DataFrame(
        {
            "desde": (huecos.index - pd.to_timedelta(huecos["paso_h"], unit="h")).round("s"),
            "hasta": huecos.index.round("s"),
            "horas": huecos["paso_h"],
        }
    ).reset_index(drop=True)
)

**Lectura final:**

Podemos observar que la lectura no es uniforme, hay lecturas casi que por minuto al inicio (cada ~1.2min), fluctuacion de segundos alrededor de la medianoche (como vimos en la sección 4) y luego un cierre diario.

La frecuencia objetivo para el modelado es diaria, construida con el cierre de cada día y con tolerancia de segundos alrededor de la medianoche, los huecos de más de 30h corresponden a días sin cierre.

## 7. Estacionalidad (train, filas `ok` con resolución diaria)

In [ ]:
d = t[t.categoria.eq("ok") & t.resolucion.eq("diaria")].copy()
d["consumo_m3"] = d["delta"]
d["fecha"] = (d.index - pd.Timedelta(seconds=5)).normalize()
d["dia_semana"] = pd.Categorical(d["fecha"].dt.dayofweek.map(dict(enumerate(DIAS))), DIAS)
d["fin_de_semana"] = np.where(d["fecha"].dt.dayofweek >= SABADO, "Fin de semana", "Entre semana")
d["mes"] = d["fecha"].dt.strftime("%Y-%m")
print(f"Días válidos para estacionalidad: {len(d)}")

fig, axes = plt.subplots(1, 2, figsize=(17, 4.5))
axes[0].bar(d["fecha"], d["consumo_m3"], color="#5BA7DE")
axes[0].plot(
    d.set_index("fecha")["consumo_m3"].asfreq("D").rolling(7, min_periods=3).mean(),
    color=INK,
    lw=2,
    label="media móvil 7 d",
)
axes[0].set_title("Consumo diario de gas (m^3, train)", loc="left", fontweight="bold", color=INK)
axes[0].legend(frameon=False)
g = d.groupby("dia_semana", observed=True)["consumo_m3"]
axes[1].errorbar(
    range(len(g.mean())),
    g.mean(),
    yerr=g.sem() * 1.96,
    marker="o",
    color="#5BA7DE",
    ecolor="#94a3b8",
    capsize=3,
)
axes[1].set_xticks(range(len(g.mean())))
axes[1].set_xticklabels(g.mean().index)
axes[1].set_title("Media por día de la semana (IC 95 %)", loc="left", fontweight="bold", color=INK)
for ax in axes:
    ax.grid(axis="y", color=GRID, ls="--")
    sns.despine(ax=ax, left=True)
plt.tight_layout()
plt.show()
display(g.agg(["count", "mean", "median"]).T)

**Lectura:**

- Se utilizaron los registros limpios, se dejó de lado los que tenían anomalías, lo que dej´p algunos espacios vacíos, en especial donde se tuvo el hueco de más de 30 horas.

- Se puede observar que el domingo es el pico de consumo de gas, con una media de 1392.25m^3, mientras que el martes es el día en el que menor consumo se da, con una media de 629.48m^3

- Podemos observar que los intervalos de confianza son demasiado grandes, lo que nos muestra que el día de la semana no es un factor muy grande para la predicción y el consumo puede variar mucho de un lunes a otro lunes por ejemplo.

- Podemos ver que la planta no tiene un ciclo exacto cada semana, lo cual muestra porque hay tanto cambio

In [ ]:
serie = d.set_index("fecha")["consumo_m3"]
serie = serie[~serie.index.duplicated()].asfreq("D")
r = acf(serie, nlags=14, missing="conservative")
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.bar(range(len(r)), r, color="#5BA7DE")
ax.axhline(0, color=INK, lw=0.8)
lim = 1.96 / np.sqrt(serie.notna().sum())
ax.axhspan(-lim, lim, color="#E2E8F0", zorder=0)
ax.set_title(
    "ACF del consumo diario (banda ≈ 95 % bajo ruido blanco)",
    loc="left",
    fontweight="bold",
    color=INK,
)
ax.set_xlabel("rezago (días)")
sns.despine(left=True)
plt.show()

for cat in ["dia_semana", "fin_de_semana", "mes"]:
    grupos = [
        x.values for _, x in d.groupby(cat, observed=True)["consumo_m3"] if len(x) >= MIN_N_GRUPO
    ]
    h, p = stats.kruskal(*grupos)
    eps2 = h / (len(d) - 1)
    print(f"Kruskal-Wallis consumo vs {cat:13s}: H = {h:5.2f}, p = {p:.3f}, ε² = {eps2:.3f}")

**Lectura:**
- El consumo no sigue un ciclo semanal claro. La única relación débil es con el día inmediatamente anterior, lo que indica que la planta tiende a mantener el ritmo de ayer.

- Hay indicios de que el domingo se consume mucho más gas (~1.390 m^3) que el resto de la semana (~630–850 m^3), pero más allá de este pico, el día exacto o el mes no hacen gran diferencia.

- Con apenas 44 días de lecturas limpias, la muestra es muy pequeña. Es imposible confirmar si estos comportamientos son reglas fijas o pura casualidad, por lo que se necesitaría de más historia para asegurarlo.


## 8. Distribuciones y outliers (train, filas `ok` diarias)


In [ ]:
desc = d["consumo_m3"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])
desc["asimetria"], desc["curtosis"] = d["consumo_m3"].skew(), d["consumo_m3"].kurt()
desc["Coeficiente de variación"] = d["consumo_m3"].std() / d["consumo_m3"].mean()
display(desc.to_frame("consumo diario (m^3)").T)

s = d["consumo_m3"]
q1, q3 = s.quantile([0.25, 0.75])
iqr = q3 - q1
mad = stats.median_abs_deviation(s, scale="normal")
d["outlier_tukey"] = (s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)
d["z_mad"] = (s - s.median()) / mad
n_tukey = int(d.outlier_tukey.sum())
n_mad = int((d.z_mad.abs() > Z_MAD_UMBRAL).sum())
print(f"Outliers Tukey: {n_tukey} · |z_MAD| > {Z_MAD_UMBRAL}: {n_mad}")
display(d.loc[d.outlier_tukey | (d.z_mad.abs() > Z_MAD_UMBRAL), ["fecha", "consumo_m3", "z_mad"]])

fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
fancy_hist(d, "consumo_m3", n_bins=10, ax=axes[0], title="Distribución del consumo diario (m^3)")
sns.boxplot(x=s, ax=axes[1], color="#5BA7DE")
axes[1].set_title("Boxplot", loc="left", fontweight="bold", color=INK)
sns.despine(ax=axes[1], left=True)
plt.tight_layout()
plt.show()

**Lectura:**
- No está fuertemente sesgada, media y mediana están cercanas y el valor de la asimetría es bajo
- Fluctua mucho (std=524.91)
- Solo se detecta un Outlier con tukey, sin embargo z_mad no lo considera atípico


In [ ]:
rep = normality_report(d, "consumo_m3")

In [ ]:
# Granularidad sub-horaria (train): caudal instantáneo, útil para entender el régimen de uso
sub = t[
    t.resolucion.eq("sub-horaria") & t.categoria.eq("granularidad") & t.paso_h.ge(0.01)
].copy()  # excluye ráfagas de segundos en medianoche
sub["caudal_m3_h"] = sub["delta"] / sub["paso_h"]
display(sub["caudal_m3_h"].describe().to_frame("caudal m^3/h (lecturas por minuto)").T)

Calculamos la "velocidad" de consumo (delta / paso_h) aprovechando las lecturas por minuto registradas al inicio de la serie. Los resultados revelan la física real del proceso:

- El flujo varía bruscamente en cuestión de minutos, saltando de 42 hasta 228 m^3/h. Esto demuestra que la planta no consume gas de forma constante, sino mediante ciclos (encendidos a máxima potencia seguidos de bajones de intensidad).

- Entender esta volatilidad minuto a minuto nos confirma que, cuando los totales diarios caen casi a cero, no es un error del sensor, sino paradas reales en la producción.

**Lectura:** 
- El consumo diario es muy variable (CV alto) y va desde casi 0 (paradas de refinación) hasta más de 2.000 m^3. 

- Los valores bajos son paradas reales, no errores. A nivel de minuto el caudal alterna entre regímenes alto y bajo, lo que confirma que el gas se usa por ciclos según la demanda del proceso.


## 9. Salud general de los datos (train)

In [ ]:
salud = pd.Series(
    {
        "filas train": len(t),
        "% filas utilizables como consumo diario (ok)": t.categoria.eq("ok").mean() * 100,
        "% faltante implícito (reinicio, hueco, sin cambio)": t.categoria.isin(
            ["reinicio", "tras_hueco", "sin_cambio"]
        ).mean()
        * 100,
        "% granularidad distinta a la diaria": t.categoria.eq("granularidad").mean() * 100,
        "chequeos de esquema que fallan": int((esquema.estado == "FALLA").sum()),
        "outliers Tukey en consumo diario": int(d.outlier_tukey.sum()),
    }
)
display(salud.to_frame("valor"))
print("Fallan:", ", ".join(esquema.loc[esquema.estado == "FALLA", "chequeo"]))

## 10. Estrategia de manejo de faltantes (documentada, **no aplicada**)

| Problema en los datos | Causa probable | Solución principal| Alternativas |
| :--- | :--- | :--- | :--- |
| Múltiples lecturas al día (por minuto o cada ciertas horas) | Pruebas de instalación o calibración inicial del sistema. | Agrupar a 1 lectura por día, tomando siempre el último registro de la medianoche (dando un margen de ±5 segundos). | Si logramos conseguir los datos agrupados por hora desde la fuente, cambiar el modelo a formato horario. |
| Medidor reiniciado a cero (`reinicio`) | Reseteo intencional o programado del totalizador. | Tomar la primera lectura después del reinicio como el consumo mínimo de ese día (cota inferior) y marcarla con el indicador `reinicio`. Para mantener continuo el acumulado, sumarle el último valor registrado antes del reinicio. | Si el modelo es sensible a estos valores estimados, excluir ese día del entrenamiento. |
| Días sin cierre (`tras_hueco`) | Falla de registro: el SCADA no guardó el cierre de uno o más días. | Para totales: repartir el consumo acumulado del hueco entre los días faltantes. Para el modelo: dejarlos vacíos (NaN) con el indicador `tras_hueco`. | Imputar con el promedio móvil de los 7 días anteriores, calculado solo con train y sin usar información posterior. |
| Primera lectura (`primera_lectura`) | No hay una lectura anterior para calcular la diferencia. | No usarla como consumo, pero **conservarla** como punto de partida para calcular el consumo del día siguiente. | N/A |
| Picos gigantes o casi nulos (Outliers) | Es la realidad física del proceso (consumo por ciclos/ráfagas térmicas). | No borrar ni "suavizar" estos datos (no son errores). Solo crear una columna que le avise al modelo cuando la planta se detiene. | Si el algoritmo sufre con variaciones tan bruscas, aplicarle una transformación matemática fuerte (como log1p). |

**Reglas de no-fuga:** las tolerancias, medias móviles y cualquier perfil se aprenden con train dentro de `fit()`. En un split cronológico ninguna imputación usa días posteriores al corte. Se conservan los indicadores de faltante como variables del modelo.

## 11. Hallazgos principales (train: 80 de 100 filas)

1. **Esquema:** las 3 columnas esperadas están presentes y con tipos correctos. Fallan 3 chequeos: el archivo viene en **orden descendente** (hay que ordenar), el **acumulado decrece** por reinicios del contador y `Consumo_Diario` tiene **valores negativos** por esos mismos reinicios. `Consumo_Diario` coincide siempre con la diferencia del acumulado.
2. **Tipos:** los dtypes ya son los esperados. La resolución del medidor es de 0,1 m^3 y los cierres diarios caen entre 23:59:56 y 00:00:02.
3. **Faltantes:** no hay NaN explícitos, pero sí **faltante implícito** en el 5 % de las filas (reinicios, un hueco de 55 h entre el 3 y el 5 de agosto, un día sin cambio). Otro 40 % de las filas tiene una **granularidad distinta a la diaria** (puesta en marcha del 1 al 3 de agosto). Solo el 55 % de las filas del train es un consumo diario directamente utilizable.
4. **Integridad temporal:** la frecuencia es mixta (minuto → ráfagas en medianoche → diaria). La frecuencia de trabajo debe ser **diaria** y construirse con el último cierre de cada día.
5. **Estacionalidad:** no es concluyente con ~2 meses. Hay un indicio de más consumo el fin de semana y no hay autocorrelación significativa.
6. **Distribución y outliers:** el consumo diario tiene mediana ≈ 936 m^3 y va de ~40 a ~2.460 m^3, con asimetría leve. Solo un día es outlier por Tukey (9-ago, 2.456 m^3) y ninguno por z-MAD. Los valores bajos son paradas de refinación.
7. **Estrategia de faltantes:** si se puede, pedir una **exportación horaria** del acumulado y aclarar con la planta la causa de los reinicios.